# PHASE 1: Data Quality Assessment (Governance First)

Reusable EDA functions for rapid data quality profiling.  
**No dataset is loaded here** — import this notebook's functions or paste a `df` into the designated cell.

In [ ]:
# --- Setup: imports and theme ---
import sys, os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats

# Add src to path so we can import the elocal theme
sys.path.insert(0, os.path.abspath('../../src'))
from elocal_analysis.elocal_theme import (
    set_elocal_theme, ELOCAL_PALETTE, ELOCAL_BLUE, ELOCAL_ORANGE, ELOCAL_GREEN
)

set_elocal_theme()
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

In [ ]:
# --- Load your dataset here ---
# Replace the path with the actual dataset when available
# DATA_PATH = os.path.join('../../data/raw/', 'your_file.csv')
# df = pd.read_csv(DATA_PATH)
# df.head()

## Completeness

In [ ]:
# --- Completeness: percent missing by column ---
def completeness_summary(df: pd.DataFrame) -> pd.DataFrame:
    """Return a DataFrame with missing count, percentage, and dtype for every column."""
    n = len(df)
    missing_count = df.isnull().sum()
    result = pd.DataFrame({
        'dtype': df.dtypes,
        'missing_count': missing_count,
        'missing_pct': (missing_count / n * 100).round(2),
        'present_pct': ((1 - missing_count / n) * 100).round(2)
    }).sort_values('missing_pct', ascending=False)
    return result

# completeness_summary(df)

In [ ]:
# --- Completeness: visualise missing values ---
def plot_missing(df: pd.DataFrame, threshold: float = 0.0) -> None:
    """Bar chart of missing-value percentage (only columns above threshold %)."""
    pct = (df.isnull().mean() * 100).sort_values(ascending=False)
    pct = pct[pct > threshold]
    if pct.empty:
        print('No columns above the missing-value threshold.')
        return
    fig, ax = plt.subplots(figsize=(16, max(4, len(pct) * 0.4)))
    sns.barplot(x=pct.values, y=pct.index, color=ELOCAL_BLUE, ax=ax)
    ax.set_xlabel('% Missing')
    ax.set_title('Missing Values by Column')
    for i, v in enumerate(pct.values):
        ax.text(v + 0.3, i, f'{v:.1f}%', va='center')
    plt.tight_layout()
    plt.show()

# plot_missing(df)

In [ ]:
# --- Completeness: missingness pattern heatmap ---
def plot_missingness_heatmap(df: pd.DataFrame) -> None:
    """Heatmap showing nullity pattern across rows & columns (sampled if large)."""
    sample = df.sample(min(500, len(df)), random_state=42) if len(df) > 500 else df
    fig, ax = plt.subplots(figsize=(16, 8))
    sns.heatmap(sample.isnull().astype(int), cbar=False,
                yticklabels=False, cmap=[ELOCAL_GREEN, ELOCAL_ORANGE], ax=ax)
    ax.set_title('Missingness Pattern (green = present, orange = missing)')
    plt.tight_layout()
    plt.show()

# plot_missingness_heatmap(df)

In [ ]:
# --- Completeness: MCAR / MAR / MNAR diagnostic ---
def missingness_diagnostic(df: pd.DataFrame) -> pd.DataFrame:
    """
    For each column with missing values, run a simple diagnostic:
      - MCAR proxy: Little's-style test — compare means of other numeric cols
        when this column IS vs IS NOT missing.
      - If significant differences exist → likely MAR or MNAR.
    Returns a summary DataFrame.
    """
    cols_with_missing = [c for c in df.columns if df[c].isnull().any()]
    numeric_cols = df.select_dtypes(include='number').columns.tolist()
    rows = []
    for col in cols_with_missing:
        mask = df[col].isnull()
        n_miss = mask.sum()
        pct = (n_miss / len(df) * 100)
        sig_cols = []
        for nc in numeric_cols:
            if nc == col:
                continue
            grp_present = df.loc[~mask, nc].dropna()
            grp_missing = df.loc[mask, nc].dropna()
            if len(grp_present) < 2 or len(grp_missing) < 2:
                continue
            _, p = stats.ttest_ind(grp_present, grp_missing, equal_var=False)
            if p < 0.05:
                sig_cols.append(nc)
        if len(sig_cols) == 0:
            pattern = 'Likely MCAR'
        else:
            pattern = f'Likely MAR (differs on: {", ".join(sig_cols[:5])})'
        rows.append({'column': col, 'missing_n': n_miss,
                      'missing_pct': round(pct, 2), 'pattern': pattern})
    return pd.DataFrame(rows)

# missingness_diagnostic(df)

## Accuracy

In [ ]:
# --- Accuracy: IQR outlier detection per column ---
def iqr_outliers(df: pd.DataFrame, col: str, factor: float = 1.5) -> pd.DataFrame:
    """Detect outliers using the IQR method for a single numeric column.
    Returns a DataFrame of outlier rows with bounds info."""
    s = df[col].dropna()
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - factor * iqr, q3 + factor * iqr
    mask = (df[col] < lower) | (df[col] > upper)
    result = df.loc[mask].copy()
    print(f"{col}: Q1={q1:.2f}  Q3={q3:.2f}  IQR={iqr:.2f}  "
          f"bounds=[{lower:.2f}, {upper:.2f}]  outliers={mask.sum()} "
          f"({mask.mean()*100:.2f}%)")
    return result

# iqr_outliers(df, 'column_name')

In [ ]:
# --- Accuracy: IQR outlier summary for all numeric columns ---
def iqr_outlier_summary(df: pd.DataFrame, factor: float = 1.5) -> pd.DataFrame:
    """Return outlier counts and percentages for every numeric column."""
    numeric_cols = df.select_dtypes(include='number').columns
    rows = []
    for col in numeric_cols:
        s = df[col].dropna()
        q1, q3 = s.quantile(0.25), s.quantile(0.75)
        iqr = q3 - q1
        lower, upper = q1 - factor * iqr, q3 + factor * iqr
        n_outliers = ((df[col] < lower) | (df[col] > upper)).sum()
        rows.append({
            'column': col, 'Q1': q1, 'Q3': q3, 'IQR': iqr,
            'lower_bound': lower, 'upper_bound': upper,
            'outlier_count': n_outliers,
            'outlier_pct': round(n_outliers / len(df) * 100, 2)
        })
    return pd.DataFrame(rows).sort_values('outlier_pct', ascending=False)

# iqr_outlier_summary(df)

In [ ]:
# --- Accuracy: box plots for outlier visualisation ---
def plot_outlier_boxplots(df: pd.DataFrame) -> None:
    """Box plots for all numeric columns to visually spot outliers."""
    numeric_cols = df.select_dtypes(include='number').columns.tolist()
    if not numeric_cols:
        print('No numeric columns.')
        return
    n_cols = min(3, len(numeric_cols))
    n_rows = -(-len(numeric_cols) // n_cols)  # ceiling division
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 5 * n_rows))
    axes = np.array(axes).flatten()
    for i, col in enumerate(numeric_cols):
        sns.boxplot(y=df[col], ax=axes[i], color=ELOCAL_BLUE)
        axes[i].set_title(f'{col}')
    for j in range(len(numeric_cols), len(axes)):
        axes[j].set_visible(False)
    plt.suptitle('Outlier Detection — Box Plots', fontsize=18, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()

# plot_outlier_boxplots(df)

In [ ]:
# --- Accuracy: impossible values & type mismatches ---
def impossible_value_check(df: pd.DataFrame) -> pd.DataFrame:
    """Flag columns where numeric values are negative (when likely shouldn't be),
    or where object columns contain mixed types."""
    rows = []
    for col in df.columns:
        issues = []
        if pd.api.types.is_numeric_dtype(df[col]):
            n_neg = (df[col] < 0).sum()
            if n_neg > 0:
                issues.append(f'{n_neg} negative values')
            n_zero = (df[col] == 0).sum()
            if n_zero > 0:
                issues.append(f'{n_zero} zeros')
        elif df[col].dtype == 'object':
            sample = df[col].dropna().head(1000)
            n_numeric_like = sample.apply(lambda x: str(x).replace('.', '', 1).replace('-', '', 1).isdigit()).sum()
            if 0 < n_numeric_like < len(sample):
                issues.append(f'mixed types ({n_numeric_like}/{len(sample)} look numeric)')
        if issues:
            rows.append({'column': col, 'dtype': str(df[col].dtype), 'issues': '; '.join(issues)})
    return pd.DataFrame(rows) if rows else pd.DataFrame(columns=['column', 'dtype', 'issues'])

# impossible_value_check(df)

## Consistency

In [ ]:
# --- Consistency: duplicate record detection ---
def duplicate_summary(df: pd.DataFrame) -> dict:
    """Count exact duplicate rows and return summary dict."""
    n_dup = df.duplicated().sum()
    result = {
        'total_rows': len(df),
        'duplicate_rows': n_dup,
        'duplicate_pct': round(n_dup / len(df) * 100, 2),
        'unique_rows': len(df) - n_dup
    }
    print(f"Duplicates: {n_dup} / {len(df)} ({result['duplicate_pct']}%)")
    return result

# duplicate_summary(df)

In [ ]:
# --- Consistency: duplicate detection on a subset of key columns ---
def duplicate_on_keys(df: pd.DataFrame, key_cols: list) -> pd.DataFrame:
    """Find rows that are duplicated on the given key columns."""
    mask = df.duplicated(subset=key_cols, keep=False)
    dups = df.loc[mask].sort_values(key_cols)
    print(f"Rows duplicated on {key_cols}: {mask.sum()} / {len(df)}")
    return dups

# duplicate_on_keys(df, ['col_a', 'col_b'])

In [ ]:
# --- Consistency: conflicting entries (same key, different values) ---
def conflicting_entries(df: pd.DataFrame, key_cols: list, value_col: str) -> pd.DataFrame:
    """Find groups sharing the same key columns but with different values
    in value_col — indicates data conflicts."""
    grouped = df.groupby(key_cols)[value_col].nunique().reset_index(name='n_unique')
    conflicts = grouped[grouped['n_unique'] > 1]
    print(f"Conflicting groups on {key_cols} for '{value_col}': {len(conflicts)}")
    return conflicts

# conflicting_entries(df, ['id_col'], 'status_col')

In [ ]:
# --- Consistency: referential integrity check ---
def referential_integrity(df: pd.DataFrame, fk_col: str,
                          ref_df: pd.DataFrame, ref_col: str) -> pd.DataFrame:
    """Check that every value in fk_col exists in ref_df[ref_col].
    Returns orphan rows (foreign key violations)."""
    ref_vals = set(ref_df[ref_col].dropna().unique())
    orphans = df[~df[fk_col].isin(ref_vals)]
    print(f"Orphan rows ({fk_col} not in reference): {len(orphans)} / {len(df)}")
    return orphans

# referential_integrity(df, 'category_id', ref_df, 'id')

## Cardinality

In [ ]:
# --- Cardinality: unique-value summary for every column ---
def cardinality_summary(df: pd.DataFrame) -> pd.DataFrame:
    """Cardinality (unique count + ratio) for each column."""
    rows = []
    for col in df.columns:
        n_unique = df[col].nunique(dropna=True)
        rows.append({
            'column': col,
            'dtype': str(df[col].dtype),
            'n_unique': n_unique,
            'cardinality_ratio': round(n_unique / len(df) * 100, 2)
        })
    return pd.DataFrame(rows).sort_values('n_unique', ascending=False)

# cardinality_summary(df)

In [ ]:
# --- Cardinality: value counts with percentage for a single column ---
def value_counts_pct(df: pd.DataFrame, col: str, top_n: int = 20) -> pd.DataFrame:
    """Value counts with count and percentage for a column."""
    vc = df[col].value_counts(dropna=False).head(top_n).reset_index()
    vc.columns = ['value', 'count']
    vc['pct'] = (vc['count'] / len(df) * 100).round(2)
    return vc

# value_counts_pct(df, 'column_name')

In [ ]:
# --- Cardinality: bar chart of value counts for a column ---
def plot_value_counts(df: pd.DataFrame, col: str, top_n: int = 20) -> None:
    """Horizontal bar chart of the top_n values in a column."""
    vc = df[col].value_counts(dropna=False).head(top_n)
    fig, ax = plt.subplots(figsize=(16, max(4, len(vc) * 0.4)))
    sns.barplot(x=vc.values, y=vc.index.astype(str), color=ELOCAL_ORANGE, ax=ax)
    ax.set_xlabel('Count')
    ax.set_title(f'Top {top_n} Values — {col}')
    for i, v in enumerate(vc.values):
        ax.text(v + 0.3, i, f'{v}', va='center')
    plt.tight_layout()
    plt.show()

# plot_value_counts(df, 'column_name')

## Univariate Analysis

In [ ]:
# --- Univariate: descriptive statistics with skewness & kurtosis ---
def univariate_stats(df: pd.DataFrame) -> pd.DataFrame:
    """Extended describe with skewness and kurtosis for numeric columns."""
    desc = df.describe().T
    desc['skew'] = df.select_dtypes(include='number').skew()
    desc['kurtosis'] = df.select_dtypes(include='number').kurtosis()
    desc['iqr'] = desc['75%'] - desc['25%']
    desc['cv'] = (desc['std'] / desc['mean']).round(4)  # coefficient of variation
    return desc

# univariate_stats(df)

In [ ]:
# --- Univariate: distribution histograms for all numeric columns ---
def plot_distributions(df: pd.DataFrame) -> None:
    """KDE + histogram for every numeric column."""
    numeric_cols = df.select_dtypes(include='number').columns.tolist()
    if not numeric_cols:
        print('No numeric columns.')
        return
    n_cols = min(3, len(numeric_cols))
    n_rows = -(-len(numeric_cols) // n_cols)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 5 * n_rows))
    axes = np.array(axes).flatten()
    for i, col in enumerate(numeric_cols):
        sns.histplot(df[col].dropna(), kde=True, ax=axes[i],
                     color=ELOCAL_BLUE, edgecolor='white')
        skew_val = df[col].skew()
        axes[i].set_title(f'{col}  (skew={skew_val:.2f})')
    for j in range(len(numeric_cols), len(axes)):
        axes[j].set_visible(False)
    plt.suptitle('Distributions — Numeric Columns', fontsize=18,
                 fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()

# plot_distributions(df)

In [ ]:
# --- Univariate: single-column distribution with mean/median lines ---
def plot_single_distribution(df: pd.DataFrame, col: str) -> None:
    """Detailed distribution plot for one column with central tendency markers."""
    data = df[col].dropna()
    fig, ax = plt.subplots(figsize=(16, 6))
    sns.histplot(data, kde=True, color=ELOCAL_BLUE, edgecolor='white', ax=ax)
    ax.axvline(data.mean(), color=ELOCAL_ORANGE, linestyle='--', label=f'Mean: {data.mean():.2f}')
    ax.axvline(data.median(), color=ELOCAL_GREEN, linestyle='-', label=f'Median: {data.median():.2f}')
    ax.legend()
    ax.set_title(f'Distribution of {col} (skew={data.skew():.2f}, kurtosis={data.kurtosis():.2f})')
    plt.tight_layout()
    plt.show()

# plot_single_distribution(df, 'column_name')

## Bivariate Analysis

For full correlation analysis including heatmap and target leakage checks, see **[EDA_Correlation.ipynb](EDA_Correlation.ipynb)**.

In [ ]:
# --- Bivariate: quick correlation table ---
def top_correlations(df: pd.DataFrame, n: int = 15) -> pd.DataFrame:
    """Return top-n absolute pairwise correlations (no self-pairs)."""
    corr = df.select_dtypes(include='number').corr()
    pairs = (
        corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
        .stack()
        .reset_index()
    )
    pairs.columns = ['feature_1', 'feature_2', 'correlation']
    pairs['abs_corr'] = pairs['correlation'].abs()
    return pairs.sort_values('abs_corr', ascending=False).head(n).drop(columns='abs_corr')

# top_correlations(df)

In [ ]:
# --- Bivariate: correlation heatmap (delegated to EDA_Correlation.ipynb for full version) ---
def plot_correlation_heatmap(df: pd.DataFrame) -> None:
    """Quick correlation heatmap for numeric columns."""
    numeric_df = df.select_dtypes(include='number')
    if numeric_df.shape[1] < 2:
        print('Need at least 2 numeric columns.')
        return
    corr = numeric_df.corr()
    mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
    fig, ax = plt.subplots(figsize=(16, 12))
    sns.heatmap(corr, mask=mask, annot=True, fmt='.2f',
                cmap='coolwarm', center=0, square=True,
                linewidths=0.5, ax=ax)
    ax.set_title('Correlation Heatmap')
    plt.tight_layout()
    plt.show()

# plot_correlation_heatmap(df)

In [ ]:
# --- Bivariate: target leakage check ---
def target_leakage_check(df: pd.DataFrame, target_col: str,
                          threshold: float = 0.95) -> pd.DataFrame:
    """Flag features with suspiciously high correlation to the target
    (absolute correlation >= threshold)."""
    numeric_df = df.select_dtypes(include='number')
    if target_col not in numeric_df.columns:
        print(f'{target_col} is not numeric — skipping leakage check.')
        return pd.DataFrame()
    corr_with_target = numeric_df.corr()[target_col].drop(target_col).abs()
    leakers = corr_with_target[corr_with_target >= threshold]
    if leakers.empty:
        print(f'No features above {threshold} correlation with {target_col}.')
    else:
        print(f'POTENTIAL LEAKAGE — features with |corr| >= {threshold}:')
        print(leakers.sort_values(ascending=False))
    return leakers.reset_index().rename(columns={'index': 'feature', target_col: 'abs_corr'})

# target_leakage_check(df, 'target_column')

---

### Quick-Run Template

Uncomment and run the cell below once you have `df` loaded to execute all governance checks at once.

In [ ]:
# --- Quick-run: execute all Phase 1 checks ---
# print('=== COMPLETENESS ===')
# display(completeness_summary(df))
# plot_missing(df)
# plot_missingness_heatmap(df)
# display(missingness_diagnostic(df))
#
# print('\n=== ACCURACY ===')
# display(iqr_outlier_summary(df))
# plot_outlier_boxplots(df)
# display(impossible_value_check(df))
#
# print('\n=== CONSISTENCY ===')
# duplicate_summary(df)
#
# print('\n=== CARDINALITY ===')
# display(cardinality_summary(df))
#
# print('\n=== UNIVARIATE ===')
# display(univariate_stats(df))
# plot_distributions(df)
#
# print('\n=== BIVARIATE ===')
# display(top_correlations(df))
# plot_correlation_heatmap(df)